# 🤖 Huấn Luyện & Chọn Checkpoint 2 Bước PPO 9D (`w_pos = 0.30`, 5s, PWM `0.60/0.40`)

**Cấu hình đồng bộ với Firmware `Project_self` (STM32L476RG):**
- **Chu kỳ điều khiển:** 200 Hz (`Ts = 5 ms`), Encoder cập nhật mỗi `20 ms` (50 Hz, 1400 CPR, bộ lọc vận tốc $N=10$).
- **Chuỗi truyền động:** Hệ số khuếch đại PPO `1.30x` → Bộ lọc PWM `0.60 × u_raw + 0.40 × u_prev` → Làm tròn số nguyên → Bù vùng chết mềm `80 PWM` (deadzone `|u| ≤ 2 → 0`).
- **Không gian quan sát 9D:** $[\theta_t, \dot{\theta}_t, x_t, v_t, \theta_{t-1}, \dot{\theta}_{t-1}, x_{t-1}, v_{t-1}, a_{t-1}]$.
- **Quy trình chọn checkpoint 2 bước mới:**
  1. Trong lúc train, giữ **Top 3 checkpoint đứng đầu bài test nhanh 6s** + **checkpoint cuối đợt train (`final_policy`)**.
  2. Sàng lọc cả 4 ứng viên trên **18 ca không nhiễu 10s + 4 ca dài không nhiễu (22 ca)**, xếp hạng ưu tiên: *ít ngã/dừng → nhiều ca giữ vị trí đạt → sai số cuối thấp*.
  3. Lấy **2 bản đứng đầu (`best_policy` & `runner_up_policy`)** chấm đủ **98 ca** và đối chiếu với bản mốc cũ `best_policy_wpos_030.npz` (`24/98` ca ngã).
- **Cổng an toàn STM32:** Bản thắng luôn lưu riêng tại `ab_winner_policy.npz` & `rl_sb3_stm32_ab_winner.h`. Chỉ ghi đè `rl_sb3_stm32.h` và `best_policy.npz` nếu đạt tiêu chí giữ vị trí (`|θ| ≤ 0.5°, |x| ≤ 2 cm` trong 2s cuối).

---
### Bước 1: Lấy code mới nhất từ GitHub, cài đặt thư viện & định nghĩa `run_step` (dừng ngay khi lỗi)

> **Lưu ý:** `git reset --hard` giữ nguyên các checkpoint cũ như `outputs/checkpoints/best_policy_wpos_030.npz` nếu bạn đang chạy tiếp trên cùng phiên Colab.

In [ ]:
import os
import sys
import subprocess

def run_step(cmd, required_files=None):
    """Chạy lệnh đồng bộ và DỪNG NGAY nếu lệnh trả về lỗi hoặc thiếu file đầu ra bắt buộc."""
    print(f"\n>>> RUN: {' '.join(cmd)}", flush=True)
    proc = subprocess.run(cmd)
    if proc.returncode != 0:
        raise RuntimeError(f"Lệnh thất bại với mã lỗi {proc.returncode}: {' '.join(cmd)}")
    if required_files:
        missing = [f for f in required_files if not os.path.exists(f)]
        if missing:
            raise FileNotFoundError(f"Thiếu file kết quả sau bước chạy: {missing}")

if os.path.exists("configs"):
    run_step(["git", "fetch", "origin"])
    run_step(["git", "reset", "--hard", "origin/p1-balance-robot"])
elif os.path.exists("self_blancing"):
    os.chdir("self_blancing")
    run_step(["git", "fetch", "origin"])
    run_step(["git", "reset", "--hard", "origin/p1-balance-robot"])
else:
    run_step(["git", "clone", "-b", "p1-balance-robot", "https://github.com/nguyenthanhsang2306/self_blancing.git"])
    os.chdir("self_blancing")

run_step([
    sys.executable, "-m", "pip", "install", "-q",
    "gymnasium", "stable-baselines3", "torch", "matplotlib", "pyyaml", "tensorboard"
])
print(f"[Ready] Thư mục làm việc hiện tại: {os.getcwd()}")
print(f"[Check] best_policy_wpos_030.npz có sẵn: {os.path.exists('outputs/checkpoints/best_policy_wpos_030.npz')}")

---
### Bước 2: Train bản 5 giây (`max_episode_steps = 1000`, `w_pos = 0.30`, `--recenter-prob 0.0`) với thưởng giữ vị trí `+0.05`

- Chỉ đổi hàm thưởng: cộng `+0.05` mỗi bước khi đồng thời `|θ| ≤ 0.5°`, `|x − x_target| ≤ 2 cm`, `|v| ≤ 1 cm/s` (`r_hold_bonus = 0.05` trong `src/environment.py`).
- Giữ nguyên lọc PWM `0.60/0.40` và cách chọn checkpoint 2 bước (`--top-k 3` + `final` → sàng lọc 22 ca không nhiễu → chấm đủ 98 ca cho Top 2).

In [ ]:
run_step(
    [
        sys.executable, "train.py",
        "--config", "configs/default.yaml",
        "--w-pos", "0.30",
        "--recenter-prob", "0.0",
        "--top-k", "3",
        "--tag", "wpos_030_5s_hold",
    ],
    required_files=[
        "outputs/checkpoints/best_policy_wpos_030_5s_hold.npz",
        "outputs/checkpoints/runner_up_policy_wpos_030_5s_hold.npz",
        "outputs/checkpoints/final_policy_wpos_030_5s_hold.npz",
        "outputs/firmware_headers/rl_sb3_stm32_wpos_030_5s_hold.h",
        "outputs/firmware_headers/rl_sb3_stm32_wpos_030_5s_hold_runner_up.h",
        "outputs/reports/eval_report_wpos_030_5s_hold.md",
    ],
)

---
### Bước 3: So sánh Top 2 bản mới (`best` & `runner_up`) với bản mốc `best_policy_wpos_030.npz` trên 98 kịch bản

- Nếu chưa có `outputs/checkpoints/best_policy_wpos_030.npz` trong Colab runtime hiện tại, ô này sẽ mở hộp thoại để bạn upload `best_policy_wpos_030.npz` trước khi chấm điểm.
- Kết quả lưu vào `outputs/reports/hold_bonus_comparison.md` và chỉ cập nhật `rl_sb3_stm32.h` nếu đạt cổng STM32 (`real_robot`).

In [ ]:
ckpt_030_prev = "outputs/checkpoints/best_policy_wpos_030.npz"
ckpt_hold_best = "outputs/checkpoints/best_policy_wpos_030_5s_hold.npz"
ckpt_hold_runner = "outputs/checkpoints/runner_up_policy_wpos_030_5s_hold.npz"

if not os.path.exists(ckpt_030_prev):
    print(f"[Notice] Chưa thấy {ckpt_030_prev} trong Colab runtime hiện tại.")
    try:
        from google.colab import files
        print("Hãy chọn file best_policy_wpos_030.npz (mốc 24/98 ngã) để upload...")
        uploaded = files.upload()
        os.makedirs("outputs/checkpoints", exist_ok=True)
        for fname, data in uploaded.items():
            dst = ckpt_030_prev if fname.endswith(".npz") else os.path.join("outputs/checkpoints", os.path.basename(fname))
            with open(dst, "wb") as uf:
                uf.write(data)
            print(f"[Upload] Đã nạp {fname} -> {dst}")
    except Exception as e:
        print(f"[Warning] Không mở được upload ({e}).")

missing_ckpts = [p for p in [ckpt_030_prev, ckpt_hold_best, ckpt_hold_runner] if not os.path.exists(p)]
if missing_ckpts:
    raise FileNotFoundError(f"Thiếu checkpoint để so sánh 3 bản: {missing_ckpts}")

run_step(
    [
        sys.executable, "inference.py",
        "--compare",
        ckpt_030_prev,
        ckpt_hold_best,
        ckpt_hold_runner,
        "--export-winner",
        "--report-out", "outputs/reports/hold_bonus_comparison.md",
    ],
    required_files=[
        "outputs/checkpoints/ab_winner_policy.npz",
        "outputs/firmware_headers/rl_sb3_stm32_ab_winner.h",
        "outputs/reports/hold_bonus_comparison.md",
    ],
)

---
### Bước 4: Vẽ đồ thị với tên ảnh riêng cho Top 2 bản mới (`best` và `runner_up`)

In [ ]:
from IPython.display import Image, display

plot_jobs = [
    ("wpos_030_5s_hold #1 | Góc +5.0° chuẩn 10s", "outputs/checkpoints/best_policy_wpos_030_5s_hold.npz", ["--angle", "5.0", "--duration", "10.0"], "outputs/figures/wpos_030_5s_hold_best_angle5_10s.png"),
    ("wpos_030_5s_hold #1 | Góc +5.0° chạy dài 25s", "outputs/checkpoints/best_policy_wpos_030_5s_hold.npz", ["--angle", "5.0", "--duration", "25.0"], "outputs/figures/wpos_030_5s_hold_best_angle5_drift25s.png"),
    ("wpos_030_5s_hold #1 | Góc 0.0° + cú đẩy 1.5 N (10s)", "outputs/checkpoints/best_policy_wpos_030_5s_hold.npz", ["--angle", "0.0", "--kick", "1.5", "--duration", "10.0"], "outputs/figures/wpos_030_5s_hold_best_angle0_kick1p5N.png"),
    ("wpos_030_5s_hold #2 | Góc +5.0° chuẩn 10s", "outputs/checkpoints/runner_up_policy_wpos_030_5s_hold.npz", ["--angle", "5.0", "--duration", "10.0"], "outputs/figures/wpos_030_5s_hold_runner_angle5_10s.png"),
    ("wpos_030_5s_hold #2 | Góc +5.0° chạy dài 25s", "outputs/checkpoints/runner_up_policy_wpos_030_5s_hold.npz", ["--angle", "5.0", "--duration", "25.0"], "outputs/figures/wpos_030_5s_hold_runner_angle5_drift25s.png"),
    ("wpos_030_5s_hold #2 | Góc 0.0° + cú đẩy 1.5 N (10s)", "outputs/checkpoints/runner_up_policy_wpos_030_5s_hold.npz", ["--angle", "0.0", "--kick", "1.5", "--duration", "10.0"], "outputs/figures/wpos_030_5s_hold_runner_angle0_kick1p5N.png"),
]

for title, ckpt, extra_args, fig_out in plot_jobs:
    if not os.path.exists(ckpt):
        raise FileNotFoundError(f"Dừng vẽ đồ thị vì thiếu checkpoint: {ckpt}")
    print(f"\n=== {title} ===", flush=True)
    run_step(
        [sys.executable, "inference.py", "--weights", ckpt, *extra_args, "--output-figure", fig_out],
        required_files=[fig_out],
    )
    display(Image(filename=fig_out))

---
### Bước 5: Kiểm tra sai số toán học xuất C Header (`PyTorch ↔ Numpy ↔ C`)

In [ ]:
run_step([sys.executable, "verify_export.py"])

---
### Bước 6: Kiểm tra đủ file kết quả, đóng gói `rl_hold_bonus_bundle.zip` & Tải về

In [ ]:
import zipfile

required_bundle_files = [
    "outputs/checkpoints/best_policy_wpos_030.npz",
    "outputs/checkpoints/best_policy_wpos_030_5s_hold.npz",
    "outputs/checkpoints/runner_up_policy_wpos_030_5s_hold.npz",
    "outputs/checkpoints/final_policy_wpos_030_5s_hold.npz",
    "outputs/checkpoints/ab_winner_policy.npz",
    "configs/default.yaml",
    "outputs/reports/hold_bonus_comparison.md",
    "outputs/reports/eval_report_wpos_030_5s_hold.md",
    "outputs/firmware_headers/rl_sb3_stm32_wpos_030_5s_hold.h",
    "outputs/firmware_headers/rl_sb3_stm32_wpos_030_5s_hold_runner_up.h",
    "outputs/firmware_headers/rl_sb3_stm32_ab_winner.h",
    "outputs/figures/wpos_030_5s_hold_best_angle5_10s.png",
    "outputs/figures/wpos_030_5s_hold_best_angle5_drift25s.png",
    "outputs/figures/wpos_030_5s_hold_best_angle0_kick1p5N.png",
    "outputs/figures/wpos_030_5s_hold_runner_angle5_10s.png",
    "outputs/figures/wpos_030_5s_hold_runner_angle5_drift25s.png",
    "outputs/figures/wpos_030_5s_hold_runner_angle0_kick1p5N.png",
]

missing_files = [p for p in required_bundle_files if not os.path.exists(p)]
if missing_files:
    raise FileNotFoundError(f"Dừng đóng gói vì thiếu file kết quả: {missing_files}")

optional_files = ["outputs/firmware_headers/rl_sb3_stm32.h"]
bundle_files = required_bundle_files + [p for p in optional_files if os.path.exists(p)]

zip_name = "rl_hold_bonus_bundle.zip"
with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fpath in bundle_files:
        zf.write(fpath, arcname=fpath)
        print(f"  + {fpath}")

print(f"\n[Bundle] Đã đóng gói thành công {len(bundle_files)} file vào {zip_name}")

try:
    from google.colab import files
    files.download(zip_name)
    files.download("outputs/reports/hold_bonus_comparison.md")
except Exception as e:
    print(f"Tự động tải bị chặn ({e}). Hãy tải thủ công từ cột trái Colab: {zip_name}")